# 优势函数


## REINFORCE 策略梯度与优势函数

### 1. REINFORCE 策略梯度

REINFORCE 算法的策略梯度公式为：

$$
\nabla_\theta J \approx \nabla_\theta \log \pi(a \mid s) \cdot G_t
$$

其中，$G_t$ 是从当前步到 episode 结束的总回报（即折扣累积回报）。

**存在的问题**：$G_t$ 的波动巨大。在同一个策略、同一个状态下，运行两次可能会得到完全不同的 $G_t$，导致梯度估计方差很大。

### 2. 引入基线 (Baseline)

为了降低方差，可以在策略梯度中减去一个基线 $V(s)$，公式变为：

$$
\nabla_\theta J \approx \nabla_\theta \log \pi(a \mid s) \cdot (G_t - V(s))
$$

括号里的 $G_t - V(s)$ 就是**优势函数（Advantage Function）**的一种估计。

### 3. 优势函数 (Advantage Function)

优势函数的正式定义为：

$$
A^\pi(s, a) = Q^\pi(s, a) - V^\pi(s) \tag{6.1}
$$

#### 符号含义说明

| 符号 | 含义 |
| :--- | :--- |
| $A^\pi(s, a)$ | **优势函数**：在状态 $s$ 下做动作 $a$，比“平均水平”好了多少。 |
| $Q^\pi(s, a)$ | **动作价值函数**：在状态 $s$ 下先做动作 $a$，之后按策略 $\pi$ 行动的期望折扣回报。 |
| $V^\pi(s)$ | **状态价值函数**：在状态 $s$ 下按策略 $\pi$ 行动的期望折扣回报。 |
| $\pi$ | **当前策略**：决定在每个状态下各动作的概率。 |

两者的差恰好表示：“因为做了动作 $a$，多拿了多少分”。

优势函数的理论定义是 A=Q−V，但实际中通常不直接计算 Q。

在状态 $s$ 做了动作 $a$ 之后，拿到的即时奖励加上下一状态的价值。如果只取一次采样（不走完整个 episode，也不对所有可能转移求平均），就得到 $Q$ 的一步估计：

$$
Q(s,a) \approx r + \gamma V(s')
$$

其中 $r$ 是这一步实际拿到的奖励，$s'$ 是这一步实际到达的下一状态。

把这个近似代入优势函数定义：

$$
A(s,a) = Q(s,a) - V(s) \approx r + \gamma V(s') - V(s)
$$

右边就是 TD Error：

$$
A(s,a) \approx r + \gamma V(s') - V(s) = \delta
$$

# 强化学习算法对比：REINFORCE vs Actor-Critic

| 对比维度 | REINFORCE (MC) | Actor-Critic (TD) |
| :--- | :--- | :--- |
| **优势估计** | $G_t - V(s)$ | $r + \gamma V(s') - V(s) = \delta$ |
| **轨迹要求** | 需要完整轨迹G_t | 走一步就更新 |
| **更新时机** | Episode 结束后 | 每走一步 |
| **方差** | 高 | 低 |
| **代价** | 无 | 需要训练 Critic |

# Actor-Critic

In [ ]:
Actor-Critic 数据流

  状态 s
    │
    ├──→ Actor（策略网络）
    │      π(a|s) → 选动作 a
    │                  │
    │              执行动作 a
    │                  │
    │                  ▼
    │              环境 → 返回 r, s'
    │                  │
    ├──→ Critic（价值网络）  │
    │      V(s)  ──────────┤
    │      V(s') ──────────┤
    │                      │
    │      δ = r + γV(s') - V(s)
    │            │
    │            ▼
    │      Actor 更新：θ ← θ + α·∇log π(a|s)·δ
    │      Critic 更新：V(s) ← V(s) + α·δ
    │
    └──→ 下一步，重复以上过程

# 离散背景下的Actor-Critic

在背景下的 actor，它的一个最终输出会有一套 softmax 在最后一层，然后用来选择不同的一个离散动作。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym
import numpy as np

# ==========================================
# 1. Actor-Critic 网络（共享特征提取层）
# ==========================================
class ActorCritic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        # 共享的特征提取层
        self.shared = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
        )
        # Actor 头：输出动作概率
        self.actor = nn.Sequential(
            nn.Linear(128, action_dim),
            nn.Softmax(dim=-1)
        )
        # Critic 头：输出状态价值
        self.critic = nn.Linear(128, 1)

    def forward(self, x):
        features = self.shared(x)
        action_probs = self.actor(features)
        state_value = self.critic(features)
        return action_probs, state_value

# ==========================================
# 2. 训练循环（每步更新，不需要等 episode 结束）
# ==========================================
env = gym.make("CartPole-v1")
model = ActorCritic(state_dim=4, action_dim=2)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
gamma = 0.99

reward_history = []

for episode in range(500):
    state, _ = env.reset()
    total_reward = 0

    while True:
        state_t = torch.FloatTensor(state)

        # Actor 选动作，Critic 评估状态
        probs, value = model(state_t)
        dist = torch.distributions.Categorical(probs)
        action = dist.sample()
        log_prob = dist.log_prob(action)

        # 执行动作
        next_state, reward, terminated, truncated, _ = env.step(action.item())
        done = terminated or truncated
        total_reward += reward

        # Critic 评估下一个状态
        with torch.no_grad():
            _, next_value = model(torch.FloatTensor(next_state))
            next_value = 0 if done else next_value

        # TD Error = 优势估计（回顾：第 6.1 节 A ≈ δ）
        td_target = reward + gamma * next_value
        td_error = td_target - value

        # Actor 损失：策略梯度 × 优势
        # 如果不detach(), 梯度会顺着 td_error 继续反向传播到 Critic 的网络参数中。这意味着，Actor 在更新自己策略的同时，会“顺便”修改 Critic 的参数。这会导致 Critic 的参数被 Actor 的梯度带偏，破坏了 Critic 作为“独立价值评估者”的客观性。
        actor_loss = -log_prob * td_error.detach()

        # Critic 损失：让 V(s) 接近 TD Target（回顾：第 6.2 节 L = δ²）
        critic_loss = td_error.pow(2)

        # 总损失
        loss = actor_loss + critic_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        state = next_state
        if done:
            break

    reward_history.append(total_reward)
    if (episode + 1) % 50 == 0:
        avg = np.mean(reward_history[-50:])
        print(f"Episode {episode+1} | Avg Reward: {avg:.1f}")

# 连续背景下的Actor-Critic

连续背景下的 Actor-Critic，Actor 会将动作输出为高斯分布，即在均值附近采样动作，同时保留一定的探索度。

第一，mu_head 输出动作均值。因为 Pendulum 的合法动作范围是 [−2,2]，所以代码用 tanh 把输出压到 [−1,1]，再乘以 2。

第二，log_std 是可学习参数。我们不直接学习 σ，而是学习 logσ，再通过 exp 得到正的标准差。这样可以避免标准差变成负数。

第三，value_head 是 Critic，输出 V(s)。同一个网络前半部分共享特征，后面分成 Actor 头和 Critic 头。这就是本章所说的 Actor-Critic：Actor 决定怎么行动，Critic 判断当前状态大概值多少钱。



In [ ]:
import torch.nn as nn
class ActorCriticContinuous(nn.Module):
    def __init__(self, state_dim=3, action_dim=1, hidden_dim=128):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.mu_head = nn.Linear(hidden_dim, action_dim)
        self.log_std = nn.Parameter(torch.zeros(action_dim))
        self.value_head = nn.Linear(hidden_dim, 1)

    def forward(self, state):
        features = self.shared(state)
        mu = torch.tanh(self.mu_head(features)) * 2.0
        std = torch.exp(self.log_std).expand_as(mu)
        value = self.value_head(features)
        return mu, std, value

AttributeError: partially initialized module 'torch' has no attribute 'distributed' (most likely due to a circular import)